# 05 · RQ3 - Transportability of the EEG model between cohorts

**RQ3a (primary):** Random Forest trained on all of BrainLat, frozen, and applied once to the Greek cohort.
**RQ3b (secondary):** Greek => BrainLat. No recalibration: the subject-level AUC is the primary metric; calibration
(slope, calibration-in-the-large, Brier) is descriptive.

Because the transported model never saw the target cohort, its score is independent of the target labels, so a
standard likelihood-ratio test of that score beyond the demographics is valid (Dinga et al., 2020): an independent
confirmation of RQ1 without the null bootstrap.

**Secondary:** the same transport after standardising each feature within its own cohort (no labels used).

**Outputs:** outputs/models/ (frozen models), outputs/predictions/transport_*.csv, outputs/results/transport.json.

**Time:** a few minutes.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

from eegad.config import load_context
from eegad.data import load_table
from eegad.reporting import fmt, fmt_ci, fmt_metrics, fmt_p, read_json, write_json

from sklearn.metrics import brier_score_loss

from eegad.models import standardize_within_cohort, transport
from eegad.stats import auc_ci, incremental_test, logit_fit, safe_logit, threshold_metrics

ctx = load_context("05")
log, out = ctx.log, ctx.out
NB, SEED, THR = ctx.n_boot, ctx.seed, ctx.threshold
df_bl, df_gr = load_table(out / "derived" / "brainlat_epochs.csv"), load_table(out / "derived" / "greek_epochs.csv")
subj_bl, subj_gr = load_table(out / "derived" / "brainlat_subjects.csv"), load_table(out / "derived" / "greek_subjects.csv")
TRANSP, TRANSP_STD, INC_IND = {}, {}, {}


def evaluate_transport(pred, subj_target, label):
    s = pred.merge(subj_target[["subject_id", "y"] + [c for c in ("Age", "sex_fem", "years_education") if c in subj_target]],
                   on="subject_id", how="inner")
    a = auc_ci(s["y"], s["p"], NB, SEED)
    Xcal = np.column_stack([np.ones(len(s)), safe_logit(s["p"])])
    cal = logit_fit(Xcal, s["y"].to_numpy(int))
    citl = logit_fit(Xcal, s["y"].to_numpy(int), fixed={1: 1.0})      # intercept with the slope fixed at 1
    res = {"label": label, "AUC": a, "n": len(s), "n_AD": int(s.y.sum()), "n_HC": int((1 - s.y).sum()),
           "calibration_slope": float(cal["beta"][1]), "citl": float(citl["beta"][0]),
           "brier": float(brier_score_loss(s["y"], s["p"])), "mean_p": float(s["p"].mean()),
           "prevalence": float(s["y"].mean()), "metrics": threshold_metrics(s["y"], s["p"], THR)}
    print(f"{label} (n = {res['n']}: {res['n_AD']} AD, {res['n_HC']} HC): AUC {fmt_ci(*a)} | calibration slope "
          f"{res['calibration_slope']:.2f} | calibration-in-the-large {res['citl']:.2f} | Brier {res['brier']:.3f} | "
          f"mean P(AD) {res['mean_p']:.2f} vs prevalence {res['prevalence']:.2f}")
    print(f"   threshold {THR}: {fmt_metrics(res['metrics'])}")
    return res, s


def transport_keys(prefix, t, with_metrics=True):
    log.keys_from(prefix, {"auc.value": t["AUC"][0], "auc.lo": t["AUC"][1], "auc.hi": t["AUC"][2],
                           "calibration_slope": t["calibration_slope"], "citl": t["citl"], "brier": t["brier"]})
    if with_metrics:
        log.keys_from(f"{prefix}.threshold", t["metrics"])


def independent_test(s, covs, label, key, one_sided):
    r = incremental_test(s.rename(columns={"p": "p_transported"}), covs, "p_transported", label)
    p_txt = f"one-sided p {fmt_p(r['p_one_sided'])}" if one_sided else f"p = {fmt_p(r['p'])}"
    print(f"{label} (n = {r['n']}): score of the other cohort's model beyond {r['covariates']}: OR/SD = "
          f"{fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)}; LR chi2(1) = {r['LR']:.2f}; {p_txt} ({r['method']})")
    log.keys_from(f"rq1.independent.{key}", {"or.value": r["OR"], "or.lo": r["OR_lo"], "or.hi": r["OR_hi"],
                                             "p_one_sided" if one_sided else "p": r["p_one_sided"] if one_sided else r["p"]})
    return r

## RQ3a - BrainLat => Greek (primary)

In [ ]:
pred = transport(df_bl, df_gr, ctx.features, "RandomForest", "RQ3a_BrainLat_to_Greek_RF_11", ctx)
TRANSP["RQ3a"], s_a = evaluate_transport(pred, subj_gr, "BrainLat → Greek")
t = TRANSP["RQ3a"]
v = log.verdict("RQ3a", "transports" if t["AUC"][1] > 0.5 else "not demonstrated")
print(f"Reading (plan): {v}")
log.record("RQ3a", "BrainLat -> Greek (frozen model): AUC",
           f"{fmt_ci(*t['AUC'])}; n = {t['n']} ({t['n_AD']} AD, {t['n_HC']} HC); calibration slope "
           f"{t['calibration_slope']:.2f}; calibration-in-the-large {t['citl']:.2f}; Brier {t['brier']:.3f}",
           "Results; Table 2", f"reading: {v}")
log.record("RQ3a", f"BrainLat -> Greek: metrics at threshold {THR}", fmt_metrics(t["metrics"]), "Table 2", "descriptive")
transport_keys("rq3a", t)
INC_IND["Greek (BrainLat model)"] = ri = independent_test(s_a, ctx.cov_greek, "Greek (BrainLat model)", "greek", True)
log.record("RQ1 (independent)", "Greek: score of the BrainLat model beyond age and sex",
           f"OR/SD = {fmt_ci(ri['OR'], ri['OR_lo'], ri['OR_hi'], 2)}; one-sided p {fmt_p(ri['p_one_sided'])} "
           "(valid parametric test: independent model)", "Results (RQ1 and RQ3)")
s_a.to_csv(out / "predictions" / "transport_RQ3a.csv", index=False)

## RQ3b - Greek => BrainLat (secondary)

In [ ]:
pred = transport(df_gr, df_bl, ctx.features, "RandomForest", "RQ3b_Greek_to_BrainLat_RF_11", ctx)
TRANSP["RQ3b"], s_b = evaluate_transport(pred, subj_bl, "Greek → BrainLat")
t = TRANSP["RQ3b"]
v = log.verdict("RQ3b", "transports" if t["AUC"][1] > 0.5 else "not demonstrated")
print(f"Reading (plan): {v}")
log.record("RQ3b", "Greek -> BrainLat (frozen model): AUC",
           f"{fmt_ci(*t['AUC'])}; n = {t['n']} ({t['n_AD']} AD, {t['n_HC']} HC); calibration slope "
           f"{t['calibration_slope']:.2f}; calibration-in-the-large {t['citl']:.2f}; Brier {t['brier']:.3f}",
           "Results; Table 2", f"reading: {v}")
log.record("RQ3b", f"Greek -> BrainLat: metrics at threshold {THR}", fmt_metrics(t["metrics"]), "Table 2", "descriptive")
transport_keys("rq3b", t)
INC_IND["BrainLat (Greek model)"] = ri = independent_test(s_b, ctx.cov_brainlat, "BrainLat (Greek model)", "brainlat", False)
log.record("RQ1 (independent)", "BrainLat: score of the Greek model beyond age, sex and education",
           f"OR/SD = {fmt_ci(ri['OR'], ri['OR_lo'], ri['OR_hi'], 2)}; p = {fmt_p(ri['p'])} "
           "(valid parametric test: independent model)", "Results (RQ1 and RQ3)")
s_b.to_csv(out / "predictions" / "transport_RQ3b.csv", index=False)

## RQ3 (secondary) - transport with the features standardised within each cohort

Each feature is standardised with the mean and SD of all epochs of its own cohort, without labels (a label-free
version of the location/scale adjustment of ComBat), before training and applying. Outside the reading rule.

In [ ]:
df_bl_z = standardize_within_cohort(df_bl, ctx.features)
df_gr_z = standardize_within_cohort(df_gr, ctx.features)
for key, src, dst, subj_target, label, name in (
        ("RQ3a", df_bl_z, df_gr_z, subj_gr, "BrainLat → Greek, standardised within cohort", "RQ3a_std_BrainLat_to_Greek"),
        ("RQ3b", df_gr_z, df_bl_z, subj_bl, "Greek → BrainLat, standardised within cohort", "RQ3b_std_Greek_to_BrainLat")):
    TRANSP_STD[key], s_ = evaluate_transport(transport(src, dst, ctx.features, "RandomForest", name, ctx),
                                             subj_target, label)
    t = TRANSP_STD[key]
    s_.to_csv(out / "predictions" / f"transport_{key}_standardised.csv", index=False)
    log.record(key, f"{label}: AUC (secondary)", f"{fmt_ci(*t['AUC'])}; calibration slope {t['calibration_slope']:.2f}; "
               f"Brier {t['brier']:.3f}", "Results (one sentence)", "outside the reading rule")
    log.keys_from(f"{key.lower()}_std", {"auc.value": t["AUC"][0], "auc.lo": t["AUC"][1], "auc.hi": t["AUC"][2],
                                         "calibration_slope": t["calibration_slope"], "citl": t["citl"]})
write_json(out / "results" / "transport.json", {"TRANSP": TRANSP, "TRANSP_STD": TRANSP_STD, "INC_IND": INC_IND})